## Configuration: Checkpoint Path (Optional)

Set this to load from a specific checkpoint. Leave empty to use latest checkpoint or start fresh.

Example: `notebooks/best_model.pt` or `checkpoints/checkpoint_epoch_5.pt`

In [ ]:
# Optional: Specify checkpoint path to load from
# Leave empty string to use latest checkpoint or start fresh
CHECKPOINT_PATH = ""  # Example: "checkpoints/checkpoint_epoch_5.pt"

if CHECKPOINT_PATH:
    print(f"Will load from specified checkpoint: {CHECKPOINT_PATH}")
else:
    print("Will use latest checkpoint or start fresh if none exists")

# VibeShift Training: Synth → Classical Transformation

This notebook trains the flow matching model to transform synthetic audio to classical style.

## Training Configuration
- **Transformation**: Synthetic → Classical
- **Data Split**: 90% training, 10% validation
- **Loss**: Flow matching with masked padding
- **Scheduler**: Warmup + Cosine annealing
- **Best Model**: Selected by validation loss

## 1. Setup and Imports

In [ ]:
import os
import re
import logging
import torch
from pathlib import Path
import glob
from datetime import datetime
import sys

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from training.dataloader import DACDataset
from torch.utils.data import DataLoader
from training.training import TrainingPipeline, TrainingConfig

print(f"Project root: {project_root}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 2. Configuration

In [ ]:
# Directory paths
ROOT = project_root
DATA_DIR = ROOT / "data" / "output"
LOG_DIR = ROOT / "logs"
CHECKPOINT_DIR = ROOT / "checkpoints"

# Training data directories
TRAIN_SYNTH_DIR = DATA_DIR / "train" / "synth_dac"
TRAIN_CLASSICAL_DIR = DATA_DIR / "train" / "classical_dac"

# Create log and checkpoint directories
LOG_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Log directory: {LOG_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 3. Setup Logging

In [ ]:
def setup_logging(log_dir: Path) -> logging.Logger:
    """Configure logging to both file and console."""
    log_dir.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = log_dir / f"training_{timestamp}.log"
    
    logger = logging.getLogger("vibeshift_training")
    logger.setLevel(logging.DEBUG)
    
    # Remove existing handlers
    logger.handlers.clear()
    
    # File handler
    file_handler = logging.FileHandler(log_file)
    file_handler.setLevel(logging.DEBUG)
    
    # Console handler
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    
    # Formatter
    formatter = logging.Formatter(
        "[%(asctime)s] %(levelname)s - %(name)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)
    
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)
    
    return logger

logger = setup_logging(LOG_DIR)
logger.info("="*60)
logger.info("VibeShift Training Notebook Started")
logger.info("="*60)

## 4. Load and Split Data (90% train, 10% validation)

In [ ]:
def setup_data(logger: logging.Logger) -> tuple:
    """Load and split data into training (90%) and validation (10%)."""
    logger.info("Setting up training and validation data...")

    # Check training directories
    if not TRAIN_SYNTH_DIR.exists():
        logger.error(f"Training synth directory not found: {TRAIN_SYNTH_DIR}")
        raise FileNotFoundError(f"Training synth directory not found: {TRAIN_SYNTH_DIR}")

    if not TRAIN_CLASSICAL_DIR.exists():
        logger.error(f"Training classical directory not found: {TRAIN_CLASSICAL_DIR}")
        raise FileNotFoundError(f"Training classical directory not found: {TRAIN_CLASSICAL_DIR}")

    # Load all files
    all_source_files = sorted(glob.glob(str(TRAIN_SYNTH_DIR / "*.pt")))
    all_target_files = sorted(glob.glob(str(TRAIN_CLASSICAL_DIR / "*.pt")))

    logger.info(f"Loaded {len(all_source_files)} source (synth) files")
    logger.info(f"Loaded {len(all_target_files)} target (classical) files")

    if not all_source_files or not all_target_files:
        logger.error("No training data files found")
        raise ValueError("No training data files found")

    # Split into 90% train, 10% validation
    split_idx = int(len(all_source_files) * 0.9)
    train_source_files = all_source_files[:split_idx]
    train_target_files = all_target_files[:split_idx]
    val_source_files = all_source_files[split_idx:]
    val_target_files = all_target_files[split_idx:]

    logger.info("Data split (90% train, 10% validation):")
    logger.info(f"Training set (Synth -> Classical transformation):")
    logger.info(f"  Source (synth): {len(train_source_files)} files")
    logger.info(f"  Target (classical): {len(train_target_files)} files")
    logger.info(f"Validation set (Synth -> Classical transformation):")
    logger.info(f"  Source (synth): {len(val_source_files)} files")
    logger.info(f"  Target (classical): {len(val_target_files)} files")

    return train_source_files, train_target_files, val_source_files, val_target_files

train_source_files, train_target_files, val_source_files, val_target_files = setup_data(logger)

## 5. Create Datasets

In [ ]:
def validate_dataset(dataset: DACDataset, logger: logging.Logger, dataset_name: str = "Dataset") -> None:
    """Validate dataset and log sample information."""
    logger.info(f"{dataset_name} created with {len(dataset)} samples")

    try:
        x0, x1, genre_id = dataset[0]
        logger.info(f"Sample shapes - x0: {x0.shape}, x1: {x1.shape}")
        logger.info(f"Sample genre ID: {genre_id}")
        logger.info(f"DAC latent_dim: {x0.shape[-1]}")
        logger.debug(f"x0 dtype: {x0.dtype}, range: [{x0.min():.4f}, {x0.max():.4f}]")
        logger.debug(f"x1 dtype: {x1.dtype}, range: [{x1.min():.4f}, {x1.max():.4f}]")
    except Exception as e:
        logger.error(f"Failed to load sample from {dataset_name}: {e}", exc_info=True)
        raise

logger.info("Creating DAC datasets...")

# Create genre IDs (classical=0 for now, rock will be added later)
train_genres = [0] * len(train_source_files)
val_genres = [0] * len(val_source_files)

train_dataset = DACDataset(
    train_source_files,
    train_target_files,
    genre_ids=train_genres,
    cache_in_memory=True,
)
validate_dataset(train_dataset, logger, "Training dataset")

val_dataset = DACDataset(
    val_source_files,
    val_target_files,
    genre_ids=val_genres,
    cache_in_memory=True,
)
validate_dataset(val_dataset, logger, "Validation dataset")

## 6. Initialize Model

In [ ]:
logger.info("Initializing training configuration...")
config = TrainingConfig()
logger.debug(f"Config: {config.__dict__}")

logger.info("Creating training pipeline...")
pipeline = TrainingPipeline(config)
logger.info("Training pipeline initialized successfully")

# Print model summary
total_params = sum(p.numel() for p in pipeline.flow.parameters())
trainable_params = sum(p.numel() for p in pipeline.flow.parameters() if p.requires_grad)
print(f"\n{'='*50}")
print(f"Model: DiT + Flow Matching")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1024**2:.2f} MB (float32)")
print(f"{'='*50}\n")

## 7. Load Checkpoint (if exists)

In [ ]:
def get_latest_checkpoint(logger: logging.Logger) -> Path | None:
    """Get latest checkpoint file based on epoch in filename."""
    if not CHECKPOINT_DIR.exists():
        logger.info(f"Checkpoint directory not found: {CHECKPOINT_DIR}")
        return None

    candidates = []
    for ckpt in CHECKPOINT_DIR.glob("checkpoint_epoch_*.pt"):
        match = re.search(r"checkpoint_epoch_(\d+)", ckpt.stem)
        if match:
            candidates.append((int(match.group(1)), ckpt))

    if not candidates:
        return None

    candidates.sort(key=lambda item: item[0])
    latest_epoch, latest_ckpt = candidates[-1]
    logger.info(f"Resuming from checkpoint: {latest_ckpt.name} (epoch {latest_epoch})")
    return latest_ckpt

start_epoch = 1
best_loss = None
latest_ckpt = get_latest_checkpoint(logger)

if latest_ckpt is not None:
    state = pipeline.load_checkpoint(str(latest_ckpt))
    last_epoch = int(state.get("epoch", 0))
    best_loss = state.get("best_loss", state.get("avg_loss", None))
    start_epoch = last_epoch + 1
    
    if start_epoch <= config.num_epochs:
        logger.info(f"Resuming training at epoch {start_epoch}")
    else:
        logger.info(
            f"Latest checkpoint is at epoch {last_epoch}; "
            f"num_epochs is {config.num_epochs}. Skipping training."
        )
else:
    logger.info("No checkpoint found. Starting fresh training.")

print(f"Start epoch: {start_epoch}")
print(f"Best loss from checkpoint: {best_loss}")

## 8. Setup Data Loaders

In [ ]:
logger.info("Setting up data loaders...")

train_loader = pipeline.setup_data(
    train_source_files,
    train_target_files,
    genre_ids=train_genres,
    cache_in_memory=False,
)

val_loader = pipeline.setup_data(
    val_source_files,
    val_target_files,
    genre_ids=val_genres,
    cache_in_memory=False,
)

logger.info("Testing batch loading...")
batch = next(iter(train_loader))
x0_batch, x1_batch, mask_batch, genre_batch = batch
logger.debug(
    f"Batch shapes - x0: {x0_batch.shape}, x1: {x1_batch.shape}, "
    f"mask: {mask_batch.shape}, genres: {genre_batch.shape}"
)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")
print(f"\nBatch shapes:")
print(f"  x0: {x0_batch.shape}")
print(f"  x1: {x1_batch.shape}")
print(f"  mask: {mask_batch.shape}")
print(f"  genres: {genre_batch.shape}")

## 9. Run Training

In [ ]:
logger.info("="*60)
logger.info("Starting training loop")
logger.info("="*60)

if start_epoch <= config.num_epochs:
    try:
        losses, val_losses = pipeline.train(
            train_loader,
            val_loader=val_loader,
            start_epoch=start_epoch,
            end_epoch=config.num_epochs,
            best_loss=best_loss,
        )
        
        logger.info("Training completed successfully")
        logger.info(f"Training losses: {losses}")
        if val_losses:
            logger.info(f"Validation losses: {val_losses}")
        logger.info(f"Final training loss: {losses[-1]:.4f}")
        if val_losses:
            logger.info(f"Final validation loss: {val_losses[-1]:.4f}")
        
        if len(losses) > 1:
            improvement = ((losses[0] - losses[-1]) / losses[0]) * 100
            logger.info(f"Training loss improvement: {improvement:.2f}%")
    except Exception as e:
        logger.error(f"Training failed: {e}", exc_info=True)
        raise
else:
    logger.info("Training already completed. Skipping.")

## 10. Verify Checkpoints

In [ ]:
def verify_checkpoints(logger: logging.Logger) -> None:
    """Verify and log saved checkpoints."""
    logger.info("="*60)
    logger.info("Verifying saved checkpoints")
    logger.info("="*60)
    
    if not CHECKPOINT_DIR.exists():
        logger.warning(f"Checkpoint directory not found: {CHECKPOINT_DIR}")
        return
    
    checkpoints = sorted(CHECKPOINT_DIR.glob("*.pt"))
    
    if not checkpoints:
        logger.warning("No checkpoints found")
        return
    
    logger.info(f"Found {len(checkpoints)} checkpoint(s)")
    
    for ckpt in checkpoints:
        try:
            size_mb = os.path.getsize(ckpt) / (1024 * 1024)
            logger.info(f"  {ckpt.name} ({size_mb:.1f} MB)")
            
            state = torch.load(ckpt, map_location='cpu')
            epoch = state.get('epoch', 'unknown')
            avg_loss = state.get('avg_loss', 'unknown')
            logger.debug(f"    Epoch: {epoch}, Average Loss: {avg_loss}")
        except Exception as e:
            logger.error(f"Failed to load checkpoint {ckpt.name}: {e}", exc_info=True)

verify_checkpoints(logger)

logger.info("="*60)
logger.info("Training notebook completed successfully!")
logger.info("="*60)

## Summary

### Training Complete ✓

**Transformation**: Synthetic audio → Classical style

**Data Split**:
- 90% training samples
- 10% validation samples

**Model**: DiT (Diffusion Transformer) + Flow Matching

**Loss Function**: Masked MSE (ignores padding)

**Learning Rate**: Warmup → Cosine decay

**Best Model Selection**: Based on validation loss

**Checkpoints saved to**: `{CHECKPOINT_DIR}`

**Logs saved to**: `{LOG_DIR}`